In [2]:
#user = r"\SagixOffice"  # HomeOffice
user = r"\vie43sq"  # University
import sys
sys.path.append(rf"C:\Users{user}\OneDrive - Universität Würzburg\GitHub\Photoswitching")

import numpy as np
import matplotlib.pyplot as plt
import src.fluorophore_systems as fs
import src.custom_plot as cp
import src.figures as fi
from IPython.display import HTML

%load_ext autoreload
%autoreload 2

In [3]:
rates = [['k_S0_S1', 7e6, "excitation", False],  
         ['k_S1_S0', 1e9, "fluorescent emission", True], 
         ['k_S1_T1', 1e6, "intersystem crossing", False],   
         ['k_T1_S0', 5e5, "vibrational relaxation", False],
         ['k_S1_S0', 1e9, "vibrational relaxation", False]]

In [4]:
system = fs.GeneralModel(number_fluorophores=2, distances=1, rates=rates)

In [5]:
system.single_states

{0: 'S0', 1: 'S1', 2: 'T1', 3: 'B'}

In [6]:
system.single_transitions

,name,rate,trivial_name,fluorescence
id,,,,
0,k_S0_S1,7.000000e+06,excitation,False
1,k_S1_S0,1.000000e+09,fluorescent emission,True
2,k_S1_T1,1.000000e+06,intersystem crossing,False
3,k_T1_S0,5.000000e+05,vibrational relaxation,False
4,k_S1_S0,1.000000e+09,vibrational relaxation,False


In [7]:
system.joined_states

,name,single_states,single_state_counter,absorbing
id,,,,
0,S0_S0,"[0, 0]","[2, 0, 0, 0]",False
1,S0_S1,"[0, 1]","[1, 1, 0, 0]",False
2,S0_T1,"[0, 2]","[1, 0, 1, 0]",False
3,S0_B,"[0, 3]","[1, 0, 0, 1]",False
4,S1_S0,"[1, 0]","[1, 1, 0, 0]",False
5,S1_S1,"[1, 1]","[0, 2, 0, 0]",False
6,S1_T1,"[1, 2]","[0, 1, 1, 0]",False
7,S1_B,"[1, 3]","[0, 1, 0, 1]",False
8,T1_S0,"[2, 0]","[1, 0, 1, 0]",False


In [8]:
system.joined_transitions

,name,joined_states_id,single_transition_id,rate,trivial_name,fluorescence
id,,,,,,
0,S0_S0__S0_S1,"(0, 1)",0,7.000000e+06,excitation,False
1,S0_S0__S1_S0,"(0, 4)",0,7.000000e+06,excitation,False
2,S0_S1__S1_S1,"(1, 5)",0,7.000000e+06,excitation,False
3,S0_T1__S1_T1,"(2, 6)",0,7.000000e+06,excitation,False
4,S0_B__S1_B,"(3, 7)",0,7.000000e+06,excitation,False
5,S1_S0__S1_S1,"(4, 5)",0,7.000000e+06,excitation,False
6,T1_S0__T1_S1,"(8, 9)",0,7.000000e+06,excitation,False
7,B_S0__B_S1,"(12, 13)",0,7.000000e+06,excitation,False
8,S0_S1__S0_S0,"(1, 0)",1,1.000000e+09,fluorescent emission,True


In [9]:
def uuu(join_trans, join_state, single_trans):
    
    counts = join_trans['joined_states_id'].value_counts()
    max_counts = counts.max()
    uni_dir = join_state.index.size
    transition_count = single_trans.index.size
    x = np.empty(shape=(uni_dir, uni_dir, transition_count))
    x[:] = 0
    altered_counts = counts.copy()
    for index_pair, transition_id, rate in zip(join_trans['joined_states_id'], join_trans['single_transition_id'], join_trans['rate']):
        x[index_pair][transition_id] = rate
    
    # normalization to yield probabilities
    
    row_sums = x.sum(axis=2)
    non_zero = np.where(row_sums != 0)
    row_sums = row_sums[:, :, np.newaxis]
    x[non_zero] = x[non_zero]/row_sums[non_zero]
    
    # sort indices and cumsum, similar to Gillespie algorithm
    
    x_sorted_indices = np.argsort(x, axis=2)
    sorted_x = np.take_along_axis(x, x_sorted_indices, axis=2)
    cum_sum_x = np.cumsum(sorted_x, axis=2)
    
    return cum_sum_x, x_sorted_indices

In [10]:
cum_sum_x, x_sorted_indices = uuu(system.joined_transitions, system.joined_states, system.single_transitions)

In [11]:
cum_sum_x

array([[[0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 1. ],
        [0. , 0. , 0. , 0. , 0. ],
        ...,
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0.5, 1. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 1. ],
        ...,
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 1. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        ...,
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ]],

       ...,

       [[0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        ...,
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 1. ],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. 

In [13]:
system.initial_row_vector

array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [14]:
system.row_sums

array([1.4000e+07, 2.0080e+09, 7.5000e+06, 7.0000e+06, 2.0080e+09,
       4.0020e+09, 2.0015e+09, 2.0010e+09, 7.5000e+06, 2.0015e+09,
       1.0000e+06, 5.0000e+05, 7.0000e+06, 2.0010e+09, 5.0000e+05,
       0.0000e+00])

In [15]:
system.simulate(n_steps=20, seed=10)

In [16]:
system.state_series

array([0, 1, 0, 1, 0, 4, 0, 4, 0, 4, 0, 4, 0, 1, 0, 4, 0, 4, 0, 1, 0],
      dtype=int64)

In [17]:
current_states = system.state_series[:-1]
future_states = system.state_series[1:]

In [19]:
specific_cum_sums = cum_sum_x[current_states, future_states, :]
specific_cum_sums

array([[0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ],
       [0. , 0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 0.5, 1. ]])

In [20]:
values_to_insert = np.random.uniform(low=0, high=1, size=len(current_states))

In [25]:
def searchsorted2d(a,b):
    # Inputs : a is (m,n) 2D array and b is (m,) 1D array.
    # Finds np.searchsorted(a[i], b[i])) in a vectorized way by
    # scaling/offsetting both inputs and then using searchsorted

    # Get scaling offset and then scale inputs
    s = np.r_[0,(np.maximum(a.max(1)-a.min(1)+1,b)+1).cumsum()[:-1]]
    a_scaled = (a+s[:,None]).ravel()
    b_scaled = b+s

    # Use searchsorted on scaled ones and then subtract offsets
    return np.searchsorted(a_scaled,b_scaled)-np.arange(len(s))*a.shape[1]

In [28]:
ttt = searchsorted2d(specific_cum_sums, values_to_insert)

In [32]:
transition_series = x_sorted_indices[current_states, future_states, ttt]

array([0, 1, 0, 4, 0, 4, 0, 1, 0, 1, 0, 1, 0, 4, 0, 4, 0, 1, 0, 4],
      dtype=int64)

In [145]:
np.concatenate((state_transition_pairs, values_to_insert.reshape(-1, 1)), axis=1)

array([[0.        , 1.        , 0.64370164],
       [1.        , 0.        , 0.06529966],
       [0.        , 1.        , 0.81442097],
       [1.        , 0.        , 0.107416  ],
       [0.        , 4.        , 0.33466409],
       [4.        , 0.        , 0.68330148],
       [0.        , 4.        , 0.40733418],
       [4.        , 0.        , 0.72952862],
       [0.        , 4.        , 0.10798355],
       [4.        , 0.        , 0.74166436],
       [0.        , 4.        , 0.81177309],
       [4.        , 0.        , 0.83216529],
       [0.        , 1.        , 0.14081199],
       [1.        , 0.        , 0.55974265],
       [0.        , 4.        , 0.07934341],
       [4.        , 0.        , 0.2398617 ],
       [0.        , 4.        , 0.21069083],
       [4.        , 0.        , 0.17080851],
       [0.        , 1.        , 0.82177176],
       [1.        , 0.        , 0.93498621]])

In [101]:
system.process()

In [102]:
system.single_state_series

array([[0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1.,
        0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 1., 0.]])